In [3]:

from pathlib import Path
import pandas as pd
import numpy as np

# Find the project folder automatically
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "expanded_model_dataset.csv"

# Load the existing dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nSeasons available:")
print(sorted(df["season"].unique()))

print("\nColumns:")
print(df.columns.tolist())

df.head()


Dataset loaded successfully!
Shape: (3458, 27)

Seasons available:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Columns:
['season', 'round', 'race_name', 'race_date', 'driver_id', 'driver_code', 'driver_number', 'driver_name', 'constructor_id', 'constructor_name', 'grid', 'position', 'position_text', 'points', 'laps', 'status', 'finish_time', 'fastest_lap_rank', 'fastest_lap_time', 'position_num', 'won_race', 'driver_prior_mean_finish', 'driver_prior_win_rate', 'driver_recent5_mean_finish', 'constructor_prior_mean_finish', 'constructor_prior_win_rate', 'starting_position_feature']


,season,round,race_name,race_date,driver_id,driver_code,driver_number,driver_name,constructor_id,constructor_name,...,fastest_lap_rank,fastest_lap_time,position_num,won_race,driver_prior_mean_finish,driver_prior_win_rate,driver_recent5_mean_finish,constructor_prior_mean_finish,constructor_prior_win_rate,starting_position_feature
0,2018,1,Australian Grand Prix,2018-03-25,alonso,ALO,14,Fernando Alonso,mclaren,McLaren,...,7.0,1:26.978,5,0,NaN,NaN,NaN,NaN,NaN,10.0
1,2018,1,Australian Grand Prix,2018-03-25,bottas,BOT,77,Valtteri Bottas,mercedes,Mercedes,...,8.0,1:27.019,8,0,NaN,NaN,NaN,NaN,NaN,15.0
2,2018,1,Australian Grand Prix,2018-03-25,brendon_hartley,HAR,28,Brendon Hartley,toro_rosso,Toro Rosso,...,13.0,1:28.176,15,0,NaN,NaN,NaN,NaN,NaN,16.0
3,2018,1,Australian Grand Prix,2018-03-25,ericsson,ERI,9,Marcus Ericsson,sauber,Sauber,...,19.0,1:32.210,19,0,NaN,NaN,NaN,NaN,NaN,17.0
4,2018,1,Australian Grand Prix,2018-03-25,gasly,GAS,10,Pierre Gasly,toro_rosso,Toro Rosso,...,18.0,1:30.649,18,0,NaN,NaN,NaN,NaN,NaN,20.0


In [8]:

# Step 1: Check the dataset before chronological validation

import pandas as pd
import numpy as np

# Confirm the data is available
print("Dataset shape:", historical_df.shape)

# Check which seasons are available
print("Seasons:", sorted(historical_df["season"].unique()))

# Check the number of races in each season
races_per_season = (
    historical_df.groupby("season")["race_name"]
    .nunique()
)

print("\nRaces per season:")
display(races_per_season.to_frame("number_of_races"))

# Check the target variable
print("\nWinner labels:")
display(historical_df["won_race"].value_counts(dropna=False))

# Check missing values in our intended features
candidate_features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

print("\nMissing values in candidate features:")
display(
    historical_df[candidate_features]
    .isna()
    .sum()
    .to_frame("missing_values")
)

print("\nPre-validation checks completed.")


Dataset shape: (3458, 27)
Seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Races per season:


,number_of_races
season,
2018,21
2019,21
2020,17
2021,22
2022,22
2023,22
2024,24
2025,24



Winner labels:


won_race
0    3285
1     173
Name: count, dtype: int64


Missing values in candidate features:


,missing_values
starting_position_feature,45
driver_prior_mean_finish,43
driver_prior_win_rate,43
driver_recent5_mean_finish,43
constructor_prior_mean_finish,32
constructor_prior_win_rate,32



Pre-validation checks completed.


In [5]:

# Display variables currently available in the notebook
%whos DataFrame


Variable   Type         Data/Info
---------------------------------
df         DataFrame    Shape: (3458, 27)


In [7]:

historical_df = df

print("Dataset shape:", historical_df.shape)
print("Seasons:", sorted(historical_df["season"].unique()))


Dataset shape: (3458, 27)
Seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [9]:

# Step 2: Define features and chronological validation seasons

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

target = "won_race"

validation_splits = [
    (2021, 2022),
    (2022, 2023),
    (2023, 2024),
]

# Keep only rows with a known target and usable race identity
data = historical_df.dropna(
    subset=[target, "season", "round", "race_name"]
).copy()

data[target] = data[target].astype(int)

print("Features:", features)
print("Target:", target)
print("Validation splits:", validation_splits)
print("Rows available:", len(data))
print("Target classes:")
print(data[target].value_counts().sort_index())


Features: ['starting_position_feature', 'driver_prior_mean_finish', 'driver_prior_win_rate', 'driver_recent5_mean_finish', 'constructor_prior_mean_finish', 'constructor_prior_win_rate']
Target: won_race
Validation splits: [(2021, 2022), (2022, 2023), (2023, 2024)]
Rows available: 3458
Target classes:
won_race
0    3285
1     173
Name: count, dtype: int64


In [10]:

# Step 3: Chronological model validation

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, brier_score_loss

results = []

# Baseline: starting position only
baseline_features = ["starting_position_feature"]

# Compare baseline, Logistic Regression, and Random Forest
for train_end, test_season in validation_splits:

    train = data[data["season"] <= train_end].copy()
    test = data[data["season"] == test_season].copy()

    print(f"\nValidating season {test_season}")
    print(f"Training rows: {len(train)}")
    print(f"Testing rows: {len(test)}")

    if train.empty or test.empty:
        print("Skipping: training or test data is empty.")
        continue

    # Ensure feature values are numeric and infinite values are missing
    for col in features:
        train[col] = pd.to_numeric(train[col], errors="coerce")
        test[col] = pd.to_numeric(test[col], errors="coerce")

    train[features] = train[features].replace([np.inf, -np.inf], np.nan)
    test[features] = test[features].replace([np.inf, -np.inf], np.nan)

    models = {
        "Baseline": (
            baseline_features,
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=2000))
            ])
        ),
        "Logistic Regression": (
            features,
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=2000))
            ])
        ),
        "Random Forest": (
            features,
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=3,
                    random_state=42,
                    n_jobs=-1
                ))
            ])
        )
    }

    for model_name, (model_features, pipeline) in models.items():

        X_train = train[model_features]
        y_train = train[target]
        X_test = test[model_features]
        y_test = test[target]

        # Train only on seasons before the test season
        pipeline.fit(X_train, y_train)

        # Probability that each driver wins
        probabilities = pipeline.predict_proba(X_test)[:, 1]

        # Binary probability metrics across driver-race records
        ll = log_loss(y_test, probabilities, labels=[0, 1])
        brier = brier_score_loss(y_test, probabilities)

        # Top-1 winner accuracy, evaluated race by race.
        # Only include races with exactly one recorded winner.
        evaluation = test[
            ["season", "round", "race_name", target]
        ].copy()

        evaluation["predicted_probability"] = probabilities

        race_winners = evaluation.groupby(
            ["season", "round"], sort=False
        )[target].sum()

        valid_races = race_winners[race_winners == 1].index

        valid_evaluation = evaluation.set_index(
            ["season", "round"]
        ).loc[
            evaluation.set_index(["season", "round"]).index.isin(
                valid_races
            )
        ].reset_index()

        if len(valid_races) > 0:
            top_predictions = valid_evaluation.loc[
                valid_evaluation.groupby(
                    ["season", "round"]
                )["predicted_probability"].idxmax()
            ]

            winner_accuracy = top_predictions[target].mean()
        else:
            winner_accuracy = np.nan

        results.append({
            "test_season": test_season,
            "model": model_name,
            "training_rows": len(train),
            "test_rows": len(test),
            "races_with_one_recorded_winner": len(valid_races),
            "top1_winner_accuracy": winner_accuracy,
            "log_loss": ll,
            "brier_score": brier
        })

# Display results
results_df = pd.DataFrame(results)

if not results_df.empty:
    results_df = results_df.sort_values(
        ["test_season", "log_loss"]
    ).reset_index(drop=True)

    print("\nCHRONOLOGICAL VALIDATION RESULTS")
    display(results_df.round(4))
else:
    print("No results generated. Check the season split and dataset.")



Validating season 2022
Training rows: 1620
Testing rows: 440

Validating season 2023
Training rows: 2060
Testing rows: 440

Validating season 2024
Training rows: 2500
Testing rows: 479

CHRONOLOGICAL VALIDATION RESULTS


,test_season,model,training_rows,test_rows,races_with_one_recorded_winner,top1_winner_accuracy,log_loss,brier_score
0,2022,Logistic Regression,1620,440,22,0.4545,0.1247,0.0363
1,2022,Baseline,1620,440,22,0.4545,0.1390,0.0367
2,2022,Random Forest,1620,440,22,0.0000,0.1391,0.0456
3,2023,Random Forest,2060,440,22,0.7273,0.0710,0.0194
4,2023,Logistic Regression,2060,440,22,0.7273,0.0793,0.0225
5,2023,Baseline,2060,440,22,0.6364,0.1173,0.0334
6,2024,Logistic Regression,2500,479,24,0.5417,0.1191,0.0346
7,2024,Random Forest,2500,479,24,0.4167,0.1199,0.0347
8,2024,Baseline,2500,479,24,0.4583,0.1287,0.0354


In [11]:

# Step 4: Race-level probability evaluation

race_results = []

for train_end, test_season in validation_splits:

    train = data[data["season"] <= train_end].copy()
    test = data[data["season"] == test_season].copy()

    # Convert feature values to numeric and handle infinities
    for col in features:
        train[col] = pd.to_numeric(train[col], errors="coerce")
        test[col] = pd.to_numeric(test[col], errors="coerce")

    train[features] = train[features].replace(
        [np.inf, -np.inf], np.nan
    )
    test[features] = test[features].replace(
        [np.inf, -np.inf], np.nan
    )

    models = {
        "Baseline": (
            ["starting_position_feature"],
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=2000))
            ])
        ),
        "Logistic Regression": (
            features,
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=2000))
            ])
        ),
        "Random Forest": (
            features,
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=3,
                    random_state=42,
                    n_jobs=-1
                ))
            ])
        )
    }

    # Evaluate only races with exactly one recorded winner
    valid_races = []

    for race_key, race in test.groupby(
        ["season", "round"], sort=True
    ):
        if race[target].sum() == 1:
            valid_races.append((race_key, race))

    for model_name, (model_features, model) in models.items():

        model.fit(train[model_features], train[target])

        raw_probabilities = model.predict_proba(
            test[model_features]
        )[:, 1]

        scored_test = test[
            ["season", "round", target]
        ].copy()

        scored_test["raw_probability"] = raw_probabilities

        top1_correct = []
        winner_log_losses = []
        multiclass_briers = []

        for race_key, race in valid_races:

            race_scores = scored_test[
                (scored_test["season"] == race_key[0]) &
                (scored_test["round"] == race_key[1])
            ].copy()

            raw = race_scores["raw_probability"].to_numpy()
            total = raw.sum()

            # Convert scores to probabilities summing to 1 per race
            if total > 0:
                probabilities = raw / total
            else:
                probabilities = np.full(
                    len(raw), 1 / len(raw)
                )

            actual = race_scores[target].to_numpy()

            predicted_winner = np.argmax(probabilities)

            top1_correct.append(
                int(actual[predicted_winner] == 1)
            )

            winner_probability = probabilities[actual == 1][0]

            winner_log_losses.append(
                -np.log(np.clip(winner_probability, 1e-15, 1.0))
            )

            multiclass_briers.append(
                np.sum((probabilities - actual) ** 2)
            )

        race_results.append({
            "test_season": test_season,
            "model": model_name,
            "races_evaluated": len(valid_races),
            "top1_winner_accuracy": np.mean(top1_correct),
            "mean_winner_log_loss": np.mean(winner_log_losses),
            "mean_multiclass_brier": np.mean(multiclass_briers)
        })

race_level_df = pd.DataFrame(race_results)

display(
    race_level_df.sort_values(
        ["test_season", "mean_winner_log_loss"]
    ).reset_index(drop=True).round(4)
)


,test_season,model,races_evaluated,top1_winner_accuracy,mean_winner_log_loss,mean_multiclass_brier
0,2022,Logistic Regression,22,0.4545,1.6013,0.7134
1,2022,Random Forest,22,0.0000,1.7791,0.8946
2,2022,Baseline,22,0.4545,1.9334,0.7344
3,2023,Random Forest,22,0.7273,0.8623,0.4037
4,2023,Logistic Regression,22,0.7273,0.9289,0.4267
5,2023,Baseline,22,0.6364,1.5440,0.6687
6,2024,Logistic Regression,24,0.5417,1.5460,0.6802
7,2024,Random Forest,24,0.4167,1.5589,0.6837
8,2024,Baseline,24,0.4583,1.7384,0.7062


In [12]:

# Step 5: Audit feature availability and missing values

feature_columns = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

print("Rows per season:")
display(
    historical_df.groupby("season").size().to_frame("rows")
)

print("\nMissing values by feature and season:")
display(
    historical_df.groupby("season")[feature_columns]
    .apply(lambda frame: frame.isna().sum())
)

print("\nCheck winner count per race:")
winner_counts = historical_df.groupby(
    ["season", "round", "race_name"]
)["won_race"].sum()

display(winner_counts.value_counts().sort_index().to_frame("race_count"))

print("\nFeature audit completed.")


Rows per season:


,rows
season,
2018,420
2019,420
2020,340
2021,440
2022,440
2023,440
2024,479
2025,479



Missing values by feature and season:


,starting_position_feature,driver_prior_mean_finish,driver_prior_win_rate,driver_recent5_mean_finish,constructor_prior_mean_finish,constructor_prior_win_rate
season,,,,,,
2018,1,20,20,20,20,20
2019,12,6,6,6,4,4
2020,5,3,3,3,2,2
2021,12,3,3,3,4,4
2022,11,2,2,2,0,0
2023,3,3,3,3,0,0
2024,1,3,3,3,2,2
2025,0,3,3,3,0,0



Check winner count per race:


,race_count
won_race,
1,173



Feature audit completed.


In [14]:

import json
from pathlib import Path

# Define the project path independently
PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

notebooks_dir = PROJECT_DIR / "notebooks"

search_terms = [
    "starting_position_feature",
    "expanded_model_dataset.csv",
]

if not notebooks_dir.exists():
    print("Notebook folder not found:", notebooks_dir)
else:
    for notebook_path in notebooks_dir.glob("*.ipynb"):
        with open(notebook_path, "r", encoding="utf-8") as file:
            notebook = json.load(file)

        for cell_number, cell in enumerate(notebook.get("cells", [])):
            source = "".join(cell.get("source", []))

            if any(term in source for term in search_terms):
                print("\n" + "=" * 70)
                print("Notebook:", notebook_path.name)
                print("Cell:", cell_number)
                print(source[:4000])



Notebook: 01_data_collection.ipynb
Cell: 0

from pathlib import Path
import pandas as pd
import joblib

# Project paths
PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

DATA_PATH = (
    PROJECT_DIR / "data" / "processed"
    / "expanded_model_dataset.csv"
)

MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

# Load the saved dataset
expanded_df = pd.read_csv(DATA_PATH)

# Restore the feature list
expanded_features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

target = "won_race"

# Restore the saved models
baseline_model = joblib.load(
    MODEL_DIR / "baseline_model.joblib"
)

enhanced_model = joblib.load(
    MODEL_DIR / "logistic_regression_model.joblib"
)

rf_model = joblib.load(
    MODEL_DIR / "random_forest_model.joblib"
)

print("Project restored successfully")

In [15]:

import json
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

for notebook_name in [
    "01_data_collection.ipynb",
    "recovery.ipynb",
]:
    notebook_path = PROJECT_DIR / "notebooks" / notebook_name

    if not notebook_path.exists():
        continue

    with open(notebook_path, "r", encoding="utf-8") as file:
        notebook = json.load(file)

    for i, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))

        if (
            "starting_position_feature" in source
            or "expanded_model_dataset.csv" in source
        ):
            print(f"\n{'=' * 60}")
            print(f"{notebook_name} — Cell {i}")
            print(source[:2500])



01_data_collection.ipynb — Cell 0

from pathlib import Path
import pandas as pd
import joblib

# Project paths
PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

DATA_PATH = (
    PROJECT_DIR / "data" / "processed"
    / "expanded_model_dataset.csv"
)

MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

# Load the saved dataset
expanded_df = pd.read_csv(DATA_PATH)

# Restore the feature list
expanded_features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

target = "won_race"

# Restore the saved models
baseline_model = joblib.load(
    MODEL_DIR / "baseline_model.joblib"
)

enhanced_model = joblib.load(
    MODEL_DIR / "logistic_regression_model.joblib"
)

rf_model = joblib.load(
    MODEL_DIR / "random_forest_model.joblib"
)

print("Project restored successfully")
print("D

In [17]:

import json
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

for notebook_path in (PROJECT_DIR / "notebooks").glob("*.ipynb"):
    with open(notebook_path, "r", encoding="utf-8") as file:
        notebook = json.load(file)

    for i, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        matches = [
            j for j, line in enumerate(lines)
            if any(term in line.lower() for term in [
                "starting_position_feature",
                "expanded_model_dataset.csv",
                "to_csv("
            ])
        ]

        if matches:
            print(f"\n--- {notebook_path.name} | Cell {i} ---")
            shown = set()

            for j in matches:
                for k in range(max(0, j - 3), min(len(lines), j + 4)):
                    if k not in shown:
                        print(f"{k + 1}: {lines[k]}")
                        shown.add(k)
                print("...")



--- 01_data_collection.ipynb | Cell 0 ---
10: 
11: DATA_PATH = (
12:     PROJECT_DIR / "data" / "processed"
13:     / "expanded_model_dataset.csv"
14: )
15: 
16: MODEL_DIR = PROJECT_DIR / "models"
...
21: 
22: # Restore the feature list
23: expanded_features = [
24:     "starting_position_feature",
25:     "driver_prior_mean_finish",
26:     "driver_prior_win_rate",
27:     "driver_recent5_mean_finish",
...

--- 01_data_collection.ipynb | Cell 7 ---
1: calendar_path = RAW_DIR / "race_calendar.csv"
2: 
3: calendar_df.to_csv(calendar_path, index=False)
4: 
5: print(f"Saved {len(calendar_df)} calendar entries to:")
6: print(calendar_path.resolve())
...

--- 01_data_collection.ipynb | Cell 13 ---
4: results_df.drop(
5:     columns=["position_num"],
6:     errors="ignore"
7: ).to_csv(results_path, index=False)
8: 
9: print("Saved:", results_path.resolve())
...

--- 01_data_collection.ipynb | Cell 15 ---
4: results_df.drop(
5:     columns=["position_num"],
6:     errors="ignore"
7: ).to_csv

In [19]:

from pathlib import Path
import pandas as pd

# Load the existing dataset directly
PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)
DATA_PATH = (
    PROJECT_DIR / "data" / "processed"
    / "expanded_model_dataset.csv"
)

expanded_df = pd.read_csv(DATA_PATH)

# Compare starting-position feature against grid
comparison = expanded_df[
    ["season", "round", "driver_name", "grid",
     "starting_position_feature"]
].copy()

comparison["grid"] = pd.to_numeric(
    comparison["grid"], errors="coerce"
)
comparison["starting_position_feature"] = pd.to_numeric(
    comparison["starting_position_feature"], errors="coerce"
)

different_rows = comparison[
    comparison["grid"].ne(
        comparison["starting_position_feature"]
    )
    | comparison["grid"].isna()
    | comparison["starting_position_feature"].isna()
]

print("Dataset loaded:", expanded_df.shape)
print("Total rows:", len(comparison))
print("Rows with differences or missing values:", len(different_rows))

display(different_rows.head(25))


Dataset loaded: (3458, 27)
Total rows: 3458
Rows with differences or missing values: 45


,season,round,driver_name,grid,starting_position_feature
182,2018,10,Brendon Hartley,0,NaN
460,2019,3,Alexander Albon,0,NaN
482,2019,4,Pierre Gasly,0,NaN
488,2019,4,Robert Kubica,0,NaN
494,2019,4,Kimi Räikkönen,0,NaN
506,2019,5,Nico Hülkenberg,0,NaN
547,2019,7,Kevin Magnussen,0,NaN
596,2019,9,George Russell,0,NaN
668,2019,13,Robert Kubica,0,NaN
694,2019,14,Kimi Räikkönen,0,NaN


In [20]:

# Summarize the mismatches without changing any data
print("Mismatch patterns:")
display(
    different_rows.groupby(
        ["grid", "starting_position_feature"],
        dropna=False
    ).size().reset_index(name="count")
)

print(
    "\nRows where grid is zero:",
    (expanded_df["grid"] == 0).sum()
)

print(
    "Rows where starting_position_feature is missing:",
    expanded_df["starting_position_feature"].isna().sum()
)


Mismatch patterns:


,grid,starting_position_feature,count
0,0,NaN,45



Rows where grid is zero: 45
Rows where starting_position_feature is missing: 45


In [21]:

import joblib
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)
MODEL_DIR = PROJECT_DIR / "models"

model_files = [
    "baseline_model.joblib",
    "logistic_regression_model.joblib",
    "random_forest_model.joblib",
]

for filename in model_files:
    model = joblib.load(MODEL_DIR / filename)

    print(f"\n{filename}")
    print("Model type:", type(model).__name__)

    if hasattr(model, "named_steps"):
        print("Pipeline steps:", list(model.named_steps.keys()))
        for step_name, step in model.named_steps.items():
            print(f"  {step_name}: {type(step).__name__}")



baseline_model.joblib
Model type: Pipeline
Pipeline steps: ['imputer', 'scaler', 'classifier']
  imputer: SimpleImputer
  scaler: StandardScaler
  classifier: LogisticRegression

logistic_regression_model.joblib
Model type: Pipeline
Pipeline steps: ['imputer', 'scaler', 'classifier']
  imputer: SimpleImputer
  scaler: StandardScaler
  classifier: LogisticRegression

random_forest_model.joblib
Model type: Pipeline
Pipeline steps: ['imputer', 'classifier']
  imputer: SimpleImputer
  classifier: RandomForestClassifier


In [22]:

import joblib
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)
MODEL_DIR = PROJECT_DIR / "models"

model_files = [
    "baseline_model.joblib",
    "logistic_regression_model.joblib",
    "random_forest_model.joblib",
]

for filename in model_files:
    model = joblib.load(MODEL_DIR / filename)

    print(f"\n{'=' * 55}")
    print(filename)

    imputer = model.named_steps["imputer"]
    print("Imputer strategy:", imputer.strategy)
    print("Missing-value indicator:", imputer.add_indicator)

    if hasattr(imputer, "statistics_"):
        print("Imputer statistics:", imputer.statistics_)



baseline_model.joblib
Imputer strategy: median
Missing-value indicator: False
Imputer statistics: [10.]

logistic_regression_model.joblib
Imputer strategy: median
Missing-value indicator: False
Imputer statistics: [10.         10.76923077  0.         11.         11.125       0.        ]

random_forest_model.joblib
Imputer strategy: median
Missing-value indicator: False
Imputer statistics: [10.         10.76923077  0.         11.         11.125       0.        ]


In [23]:

# Read-only checks before chronological validation

print("Rows:", len(expanded_df))
print("Seasons:", sorted(expanded_df["season"].unique()))

race_winner_check = (
    expanded_df.groupby(["season", "round"])["won_race"]
    .agg(runners="size", winners="sum")
    .reset_index()
)

print("\nRace count:", len(race_winner_check))
print(
    "Races with exactly one winner:",
    (race_winner_check["winners"] == 1).sum(),
)
print(
    "Races with zero or multiple winners:",
    (race_winner_check["winners"] != 1).sum(),
)

display(
    race_winner_check[
        race_winner_check["winners"] != 1
    ].head(20)
)


Rows: 3458
Seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Race count: 173
Races with exactly one winner: 173
Races with zero or multiple winners: 0


,season,round,runners,winners


In [24]:

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Features and target already used in the project
features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]
target = "won_race"

# Evaluate each season using only earlier seasons for training
results = []

for test_season in range(2021, 2026):
    train = expanded_df[expanded_df["season"] < test_season].copy()
    test = expanded_df[expanded_df["season"] == test_season].copy()

    X_train = train[features]
    y_train = train[target].astype(int)
    X_test = test[features]
    y_test = test[target].astype(int)

    models = {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=2000, class_weight="balanced"
            )),
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1,
            )),
        ]),
    }

    # Baseline: starting position only
    baseline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000, class_weight="balanced"
        )),
    ])
    models["Baseline"] = baseline

    for model_name, model in models.items():
        selected_features = (
            ["starting_position_feature"]
            if model_name == "Baseline"
            else features
        )

        model.fit(train[selected_features], y_train)
        probabilities = model.predict_proba(
            test[selected_features]
        )[:, 1]

        # Normalize driver winner scores within each race
        race_results = test[
            ["season", "round", target]
        ].copy()
        race_results["probability"] = probabilities

        race_results["race_probability"] = (
            race_results.groupby(["season", "round"])["probability"]
            .transform(
                lambda s: s / s.sum() if s.sum() > 0 else 1 / len(s)
            )
        )

        # Top-1 winner accuracy: did the highest-scored driver win?
        race_winners = race_results.loc[
            race_results.groupby(["season", "round"])["race_probability"]
            .idxmax()
        ]
        top1_accuracy = race_winners[target].mean()

        # Race-level multiclass log loss and Brier score
        race_log_losses = []
        race_brier_scores = []

        for _, group in race_results.groupby(["season", "round"]):
            y_race = group[target].to_numpy()
            p_race = group["race_probability"].to_numpy()
            p_race = np.clip(p_race, 1e-15, 1)
            p_race = p_race / p_race.sum()

            race_log_losses.append(
                -np.sum(y_race * np.log(p_race))
            )
            race_brier_scores.append(
                np.sum((p_race - y_race) ** 2)
            )

        results.append({
            "test_season": test_season,
            "model": model_name,
            "training_seasons": f"2018–{test_season - 1}",
            "races": test[["season", "round"]].drop_duplicates().shape[0],
            "top1_winner_accuracy": top1_accuracy,
            "mean_race_log_loss": np.mean(race_log_losses),
            "mean_race_brier_score": np.mean(race_brier_scores),
        })

validation_results = pd.DataFrame(results)

display(
    validation_results.sort_values(
        ["test_season", "mean_race_log_loss"]
    ).round(4)
)


,test_season,model,training_seasons,races,top1_winner_accuracy,mean_race_log_loss,mean_race_brier_score
1,2021,Random Forest,2018–2020,22,0.2727,1.7520,0.7827
0,2021,Logistic Regression,2018–2020,22,0.3636,1.7524,0.7586
2,2021,Baseline,2018–2020,22,0.5000,1.9412,0.8097
4,2022,Random Forest,2018–2021,22,0.2273,1.8047,0.9221
3,2022,Logistic Regression,2018–2021,22,0.4091,1.8427,0.7956
5,2022,Baseline,2018–2021,22,0.4545,2.1103,0.8222
7,2023,Random Forest,2018–2022,22,0.6818,0.8371,0.4172
6,2023,Logistic Regression,2018–2022,22,0.7273,1.5517,0.7124
8,2023,Baseline,2018–2022,22,0.6364,1.9172,0.8100
9,2024,Logistic Regression,2018–2023,24,0.4583,1.7693,0.7834


In [25]:

# Diagnostic: compare baseline predictions with actual pole-sitter wins

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

baseline_features = ["starting_position_feature"]
diagnostics = []

for season in range(2021, 2026):
    train = expanded_df[expanded_df["season"] < season].copy()
    test = expanded_df[expanded_df["season"] == season].copy()

    baseline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )),
    ])

    baseline.fit(train[baseline_features], train["won_race"])
    test = test.copy()
    test["baseline_probability"] = baseline.predict_proba(
        test[baseline_features]
    )[:, 1]

    # Select the highest-scoring driver in each race
    picks = test.loc[
        test.groupby(["season", "round"])["baseline_probability"].idxmax()
    ]

    # Actual wins by drivers who started from grid position 1
    grid_numeric = pd.to_numeric(test["grid"], errors="coerce")
    pole_rows = test[grid_numeric == 1]

    diagnostics.append({
        "season": season,
        "races": test["round"].nunique(),
        "baseline_correct_picks": int(picks["won_race"].sum()),
        "baseline_accuracy": picks["won_race"].mean(),
        "pole_sitter_actual_wins": int(pole_rows["won_race"].sum()),
        "baseline_picks_match_pole": bool(
            set(picks.index) == set(pole_rows.index)
        ) if len(picks) == len(pole_rows) else False,
    })

display(pd.DataFrame(diagnostics).round(4))


,season,races,baseline_correct_picks,baseline_accuracy,pole_sitter_actual_wins,baseline_picks_match_pole
0,2021,22,11,0.5000,11,True
1,2022,22,10,0.4545,10,True
2,2023,22,14,0.6364,14,True
3,2024,24,11,0.4583,11,True
4,2025,24,16,0.6667,16,True


In [26]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\saksham\Desktop\Project\f1-prediction-platform")
REPORTS_DIR = PROJECT_DIR / "reports"

files_to_check = [
    "model_comparison_2025_rebuilt.csv",
    "model_comparison_2025.csv",
    "probability_evaluation_2025_normalized.csv",
    "probability_evaluation_2025.csv",
    "normalized_summary.csv",
    "model metrics.csv",
    "normalized metrics.csv",
]

for filename in files_to_check:
    filepath = REPORTS_DIR / filename

    if filepath.exists():
        df = pd.read_csv(filepath)
        print(f"\n{'=' * 70}")
        print(f"FILE: {filename}")
        print(f"Shape: {df.shape}")
        print("Columns:", df.columns.tolist())
        display(df.head(5))
    else:
        print(f"\nNot found: {filename}")



FILE: model_comparison_2025_rebuilt.csv
Shape: (3, 6)
Columns: ['model', 'races_evaluated', 'top1_correct', 'top1_accuracy_pct', 'winner_in_top3_pct', 'mean_actual_winner_rank']


,model,races_evaluated,top1_correct,top1_accuracy_pct,winner_in_top3_pct,mean_actual_winner_rank
0,Starting-grid baseline,24,16,66.67,95.83,1.54
1,Logistic Regression + history,24,12,50.00,100.00,1.67
2,Random Forest + history,24,14,58.33,87.50,1.79



FILE: model_comparison_2025.csv
Shape: (3, 6)
Columns: ['model', 'races', 'correct_winners', 'top1_accuracy_pct', 'winner_in_top3_pct', 'mean_winner_rank']


,model,races,correct_winners,top1_accuracy_pct,winner_in_top3_pct,mean_winner_rank
0,Baseline: starting grid,24,16,66.67,95.83,1.54
1,Logistic regression + history,24,12,50.00,100.00,1.67
2,Random Forest + history,24,14,58.33,87.50,1.79



FILE: probability_evaluation_2025_normalized.csv
Shape: (4, 5)
Columns: ['model', 'races_evaluated', 'mean_log_loss', 'mean_brier_score', 'mean_winner_probability']


...


FILE: probability_evaluation_2025.csv
Shape: (4, 5)
Columns: ['model', 'races_evaluated', 'mean_log_loss', 'mean_brier_score', 'mean_winner_probability']


,model,races_evaluated,mean_log_loss,mean_brier_score,mean_winner_probability
0,Baseline,24,2.737363,0.920696,0.064998
1,Logistic Regression,24,2.701290,0.915369,0.067818
2,Random Forest,24,2.697920,0.914387,0.068398
3,Uniform baseline,24,2.993595,0.949890,0.050110



FILE: normalized_summary.csv
Shape: (3, 3)
Columns: ['mean_log_loss', 'mean_brier_score', 'mean_winner_probability']


,mean_log_loss,mean_brier_score,mean_winner_probability
0,1.1693,0.5980,0.3442
1,1.2254,0.6138,0.3114
2,1.3708,0.6585,0.2878



Not found: model metrics.csv

Not found: normalized metrics.csv


In [27]:

from pathlib import Path
import pandas as pd

REPORTS_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform\reports"
)

for filename in [
    "predictions_2025_rebuilt.csv",
    "predictions_2025.csv",
    "race_level_probabilities_2025_normalized.csv",
    "race_level_probabilities_2025.csv",
]:
    path = REPORTS_DIR / filename

    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{'=' * 60}\n{filename}")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(3))
    else:
        print(f"\nMissing file: {filename}")




predictions_2025_rebuilt.csv
Shape: (479, 15)
Columns: ['season', 'round', 'race_name', 'driver_id', 'driver_name', 'constructor_name', 'grid', 'starting_position_feature', 'won_race', 'baseline_score', 'logistic_score', 'random_forest_score', 'baseline_rank', 'logistic_rank', 'rf_rank']


,season,round,race_name,driver_id,driver_name,constructor_name,grid,starting_position_feature,won_race,baseline_score,logistic_score,random_forest_score,baseline_rank,logistic_rank,rf_rank
0,2025,1,Australian Grand Prix,albon,Alexander Albon,Williams,6,6.0,0,0.032194,0.002932,0.022717,6,10,7
1,2025,1,Australian Grand Prix,alonso,Fernando Alonso,Aston Martin,12,12.0,0,0.001062,0.000928,0.000179,12,12,17
2,2025,1,Australian Grand Prix,antonelli,Andrea Kimi Antonelli,Mercedes,16,16.0,0,0.000107,0.000122,0.006285,16,16,11



predictions_2025.csv
Shape: (479, 14)
Columns: ['season', 'round', 'race_name', 'driver_id', 'driver_name', 'constructor_name', 'grid', 'won_race', 'baseline_score', 'logistic_score', 'random_forest_score', 'baseline_rank', 'logistic_rank', 'rf_rank']


,season,round,race_name,driver_id,driver_name,constructor_name,grid,won_race,baseline_score,logistic_score,random_forest_score,baseline_rank,logistic_rank,rf_rank
0,2025,1,Australian Grand Prix,albon,Alexander Albon,Williams,6,0,0.032194,0.002932,0.022717,6.0,10.0,7.0
1,2025,1,Australian Grand Prix,alonso,Fernando Alonso,Aston Martin,12,0,0.001062,0.000928,0.000179,12.0,12.0,17.0
2,2025,1,Australian Grand Prix,antonelli,Andrea Kimi Antonelli,Mercedes,16,0,0.000107,0.000122,0.006285,16.0,16.0,11.0



race_level_probabilities_2025_normalized.csv
Shape: (479, 18)
Columns: ['season', 'round', 'race_name', 'driver_id', 'driver_name', 'constructor_name', 'grid', 'starting_position_feature', 'won_race', 'baseline_score', 'logistic_score', 'random_forest_score', 'baseline_rank', 'logistic_rank', 'rf_rank', 'baseline_normalized_probability', 'logistic_normalized_probability', 'rf_normalized_probability']


,season,round,race_name,driver_id,driver_name,constructor_name,grid,starting_position_feature,won_race,baseline_score,logistic_score,random_forest_score,baseline_rank,logistic_rank,rf_rank,baseline_normalized_probability,logistic_normalized_probability,rf_normalized_probability
0,2025,1,Australian Grand Prix,albon,Alexander Albon,Williams,6,6.0,0,0.032194,0.002932,0.022717,6,10,7,0.032193,0.003095,0.019746
1,2025,1,Australian Grand Prix,alonso,Fernando Alonso,Aston Martin,12,12.0,0,0.001062,0.000928,0.000179,12,12,17,0.001062,0.000979,0.000155
2,2025,1,Australian Grand Prix,antonelli,Andrea Kimi Antonelli,Mercedes,16,16.0,0,0.000107,0.000122,0.006285,16,16,11,0.000107,0.000129,0.005463



race_level_probabilities_2025.csv
Shape: (479, 17)
Columns: ['season', 'round', 'race_name', 'driver_id', 'driver_name', 'constructor_name', 'grid', 'won_race', 'baseline_score', 'logistic_score', 'random_forest_score', 'baseline_rank', 'logistic_rank', 'rf_rank', 'baseline_win_probability', 'logistic_win_probability', 'rf_win_probability']


,season,round,race_name,driver_id,driver_name,constructor_name,grid,won_race,baseline_score,logistic_score,random_forest_score,baseline_rank,logistic_rank,rf_rank,baseline_win_probability,logistic_win_probability,rf_win_probability
0,2025,1,Australian Grand Prix,albon,Alexander Albon,Williams,6,0,0.032194,0.002932,0.022717,6.0,10.0,7.0,0.048872,0.047610,0.048009
1,2025,1,Australian Grand Prix,alonso,Fernando Alonso,Aston Martin,12,0,0.001062,0.000928,0.000179,12.0,12.0,17.0,0.047374,0.047515,0.046939
2,2025,1,Australian Grand Prix,antonelli,Andrea Kimi Antonelli,Mercedes,16,0,0.000107,0.000122,0.006285,16.0,16.0,11.0,0.047329,0.047477,0.047227


In [28]:

from pathlib import Path
import pandas as pd

REPORTS_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform\reports"
)

files = {
    "normalized": (
        "race_level_probabilities_2025_normalized.csv",
        [
            "baseline_normalized_probability",
            "logistic_normalized_probability",
            "rf_normalized_probability",
        ],
    ),
    "older": (
        "race_level_probabilities_2025.csv",
        [
            "baseline_win_probability",
            "logistic_win_probability",
            "rf_win_probability",
        ],
    ),
}

for label, (filename, probability_columns) in files.items():
    path = REPORTS_DIR / filename
    df = pd.read_csv(path)

    print(f"\n{'=' * 60}")
    print(f"{label.upper()} FILE: {filename}")

    for column in probability_columns:
        if column not in df.columns:
            print(f"{column}: column missing")
            continue

        race_sums = df.groupby(
            ["season", "round", "race_name"]
        )[column].sum(min_count=1)

        print(f"\n{column}")
        print("Race probability sums — first 5:")
        print(race_sums.head().round(6).to_string())
        print("Minimum race sum:", round(race_sums.min(), 6))
        print("Maximum race sum:", round(race_sums.max(), 6))
        print("Races summing to 1 (tolerance 0.001):",
              int(((race_sums - 1).abs() <= 0.001).sum()),
              "of", len(race_sums))



NORMALIZED FILE: race_level_probabilities_2025_normalized.csv

baseline_normalized_probability
Race probability sums — first 5:
season  round  race_name               
2025    1      Australian Grand Prix       1.0
        2      Chinese Grand Prix          1.0
        3      Japanese Grand Prix         1.0
        4      Bahrain Grand Prix          1.0
        5      Saudi Arabian Grand Prix    1.0
Minimum race sum: 1.0
Maximum race sum: 1.0
Races summing to 1 (tolerance 0.001): 24 of 24

logistic_normalized_probability
Race probability sums — first 5:
season  round  race_name               
2025    1      Australian Grand Prix       1.0
        2      Chinese Grand Prix          1.0
        3      Japanese Grand Prix         1.0
        4      Bahrain Grand Prix          1.0
        5      Saudi Arabian Grand Prix    1.0
Minimum race sum: 1.0
Maximum race sum: 1.0
Races summing to 1 (tolerance 0.001): 24 of 24

rf_normalized_probability
Race probability sums — first 5:
season  round

In [29]:

from pathlib import Path
import pandas as pd

REPORTS_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform\reports"
)

df = pd.read_csv(
    REPORTS_DIR / "race_level_probabilities_2025_normalized.csv"
)

# Inspect the first 5 races and each model's top-ranked driver.
probability_columns = {
    "Baseline": ("baseline_normalized_probability", "baseline_rank"),
    "Logistic Regression": ("logistic_normalized_probability", "logistic_rank"),
    "Random Forest": ("rf_normalized_probability", "rf_rank"),
}

for (season, rnd, race_name), race in df.groupby(
    ["season", "round", "race_name"], sort=True
):
    if rnd > 5:
        break

    winners = race[race["won_race"] == 1]
    print(f"\n{rnd}. {race_name}")

    if winners.empty:
        print("Winner label missing; skipping.")
        continue

    print("Actual winner:", winners.iloc[0]["driver_name"])

    for model, (prob_col, rank_col) in probability_columns.items():
        top = race.sort_values(rank_col).iloc[0]
        winner = winners.iloc[0]

        print(
            f"  {model}: predicts {top['driver_name']} "
            f"({top[prob_col]:.1%}); "
            f"actual winner probability = {winner[prob_col]:.1%}"
        )



1. Australian Grand Prix
Actual winner: Lando Norris
  Baseline: predicts Lando Norris (37.0%); actual winner probability = 37.0%
  Logistic Regression: predicts Max Verstappen (34.3%); actual winner probability = 29.9%
  Random Forest: predicts Lando Norris (33.2%); actual winner probability = 33.2%

2. Chinese Grand Prix
Actual winner: Oscar Piastri
  Baseline: predicts Oscar Piastri (37.0%); actual winner probability = 37.0%
  Logistic Regression: predicts Max Verstappen (29.0%); actual winner probability = 19.0%
  Random Forest: predicts Max Verstappen (26.0%); actual winner probability = 21.9%

3. Japanese Grand Prix
Actual winner: Max Verstappen
  Baseline: predicts Max Verstappen (37.0%); actual winner probability = 37.0%
  Logistic Regression: predicts Max Verstappen (47.3%); actual winner probability = 47.3%
  Random Forest: predicts Max Verstappen (40.6%); actual winner probability = 40.6%

4. Bahrain Grand Prix
Actual winner: Oscar Piastri
  Baseline: predicts Oscar Piastri